# Day 19: Asynchronous Python & OpenAI Agents SDK

Welcome to Day 19! Today we'll learn about:
- Asynchronous Python (async/await)
- OpenAI Agents SDK basics
- Building multi-agent workflows
- Tools and handoffs
- Creating an automated Sales Development Rep

Let's get started!


## Part 1: Understanding Async IO

Before we dive into OpenAI Agents SDK, we need to understand asynchronous Python.
All agent frameworks use async IO, so this is essential knowledge.


In [1]:
# Import necessary libraries
import asyncio
from dotenv import load_dotenv
import os


### Simple Async Example

Let's start with a basic example to understand how async/await works.


In [2]:
# This is a coroutine (async function)
async def do_some_processing():
    """Simulates some processing work"""
    print("Starting processing...")
    await asyncio.sleep(2)  # Simulates waiting (like an API call)
    print("Processing complete!")
    return "done"

# To run it, we need to await it
result = await do_some_processing()
print(f"Result: {result}")


Starting processing...
Processing complete!
Result: done


**Important**: Calling `do_some_processing()` without `await` just returns a coroutine object.
It doesn't actually run! You must use `await` to execute it.


### Running Multiple Coroutines Concurrently

The power of async IO comes from running multiple operations at once using `asyncio.gather()`.


In [3]:
async def fetch_data(id):
    """Simulates fetching data from an API"""
    print(f"Starting to fetch data for {id}")
    await asyncio.sleep(2)  # Simulates network delay
    print(f"Finished fetching data for {id}")
    return f"Data for {id}"

# Run three fetch operations concurrently
# This will take ~2 seconds total, not 6!
results = await asyncio.gather(
    fetch_data(1),
    fetch_data(2),
    fetch_data(3)
)

print(f"\nAll results: {results}")


Starting to fetch data for 1
Starting to fetch data for 2
Starting to fetch data for 3
Finished fetching data for 1
Finished fetching data for 2
Finished fetching data for 3

All results: ['Data for 1', 'Data for 2', 'Data for 3']


Notice how all three started at nearly the same time, and all finished together!
This is the magic of async IO - perfect for making multiple LLM API calls.


## Part 2: Getting Started with OpenAI Agents SDK (Swarm)

Now let's dive into the OpenAI Agents SDK!

**Important Note:** The OpenAI Agents SDK is also known as "Swarm". The transcript mentions `agents` package with `runner` and `trace`, but the actual implementation uses the `swarm` package with a `Swarm` client. The concepts are the same, just slightly different API.

### Step 1: Import Required Libraries


In [4]:
# Import from the swarm package (OpenAI Swarm/Agents SDK)
from swarm import Swarm, Agent

# For SendGrid email functionality (we'll use this later)
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail

# Load environment variables (if not already loaded)
from dotenv import load_dotenv
load_dotenv(override=True)

# Create Swarm client (uses OPENAI_API_KEY from environment)
client = Swarm()

print("✓ Swarm client initialized")


✓ Swarm client initialized


### Step 2: Load Environment Variables

Make sure your `.env` file contains:
- OPENAI_API_KEY
- SENDGRID_API_KEY (we'll use this later)


In [5]:
# Load environment variables
load_dotenv(override=True)

# Verify OpenAI key is loaded
openai_key = os.getenv("OPENAI_API_KEY")
if openai_key:
    print(f"✓ OpenAI API key loaded (starts with: {openai_key[:10]}...)")
else:
    print("✗ OpenAI API key not found. Please check your .env file.")


✓ OpenAI API key loaded (starts with: sk-proj-Tk...)


### Step 3: Create Your First Agent

Let's create a simple joke-telling agent!


In [6]:
# Create an agent
jokester = Agent(
    name="jokester",
    instructions="You are a joke teller.",
    model="gpt-4o-mini"  # You can use different models
)

print("✓ Jokester agent created")
print(jokester)


✓ Jokester agent created
name='jokester' model='gpt-4o-mini' instructions='You are a joke teller.' functions=[] tool_choice=None parallel_tool_calls=True


### Step 4: Run the Agent


In [7]:
# Run the agent using Swarm client
response = client.run(
    agent=jokester,
    messages=[{"role": "user", "content": "Tell a joke about autonomous AI agents"}]
)

# Print the response
print(response.messages[-1]["content"])


Why did the autonomous AI agent get kicked out of the party?

Because it kept trying to optimize everyone's dance moves!


### Step 5: Monitoring Agent Calls

You can monitor and debug your agent interactions in the OpenAI dashboard.
View API calls at: https://platform.openai.com


In [8]:
# Run the agent (Swarm doesn't have built-in trace, but you can see calls in OpenAI dashboard)
response = client.run(
    agent=jokester,
    messages=[{"role": "user", "content": "Tell a joke about autonomous AI agents"}]
)

# Print the response
print(response.messages[-1]["content"])

print("\n✓ You can view API calls in the OpenAI dashboard at: https://platform.openai.com")


Why did the autonomous AI agent break up with its partner?

Because it just needed some space to learn and grow—without any emotional baggage!

✓ You can view API calls in the OpenAI dashboard at: https://platform.openai.com


## Part 3: Building a Multi-Agent Workflow

Now let's build something more interesting: an automated Sales Development Rep!
We'll create multiple agents with different writing styles.

### Step 6: Create Three Sales Agents with Different Styles


In [9]:
# Instructions for three different styles
professional_instructions = """
You are a sales agent working for CompliAI, a SaaS tool that ensures 
SOC 2 compliance for companies. You write professional, serious cold emails.
"""

engaging_instructions = """
You are a humorous, engaging sales agent working for CompliAI, a SaaS tool 
that ensures SOC 2 compliance. You write witty, engaging cold emails that 
are likely to get a response.
"""

concise_instructions = """
You are a busy sales agent working for CompliAI, a SaaS tool that ensures 
SOC 2 compliance. You write concise, to-the-point cold emails.
"""

# Create three agents
sales_agent_1 = Agent(
    name="professional_sales_agent",
    instructions=professional_instructions,
    model="gpt-4o-mini"
)

sales_agent_2 = Agent(
    name="engaging_sales_agent",
    instructions=engaging_instructions,
    model="gpt-4o-mini"
)

sales_agent_3 = Agent(
    name="concise_sales_agent",
    instructions=concise_instructions,
    model="gpt-4o-mini"
)

print("✓ Three sales agents created")


✓ Three sales agents created


### Step 7: Run All Three Agents

Let's generate three different cold emails from our three agents.


In [11]:
# Run all three agents sequentially
message = "Write a cold sales email"

# Generate three emails from different agents
emails = []
for i, agent in enumerate([sales_agent_1, sales_agent_2, sales_agent_3], 1):
    print(f"Generating email {i}...")
    response = client.run(
        agent=agent,
        messages=[{"role": "user", "content": message}]
    )
    email = response.messages[-1]["content"]
    emails.append(email)
    
    print(f"\n{'='*60}")
    print(f"EMAIL {i}")
    print(f"{'='*60}")
    print(email)


Generating email 1...

EMAIL 1
Subject: Strengthen Your Compliance with CompliAI

Dear [Recipient's Name],

I hope this message finds you well.

As data privacy and security concerns continue to rise, achieving and maintaining SOC 2 compliance has become increasingly crucial for companies looking to build trust with their clients and partners. At CompliAI, we specialize in streamlining the compliance process, empowering organizations like yours to navigate these complex requirements with ease.

Our SaaS tool offers:

- **Automated Compliance Tracking**: Monitor your compliance status in real-time, minimizing the risk of oversights.
- **Customizable Workflows**: Tailor compliance processes to fit your unique business needs.
- **Comprehensive Reporting**: Generate detailed reports to keep stakeholders informed and demonstrate your commitment to security.

We understand that implementing a robust compliance strategy can be daunting. That’s why our solution is designed to simplify the proc

### Step 8: Create a Picker Agent

Now let's create an agent that picks the best email from the three options.


In [12]:
# Create a sales picker agent
sales_picker = Agent(
    name="sales_picker",
    instructions="""
    Pick the best cold sales email from the given options.
    Imagine you are a customer. Pick the one you're most likely to respond to.
    Don't give an explanation. Reply with the email you select only.
    """,
    model="gpt-4o-mini"
)

print("✓ Sales picker agent created")


✓ Sales picker agent created


### Step 9: Complete Workflow - Generate and Pick

Let's put it all together: generate three emails, then pick the best one.


In [13]:
# Complete workflow
message = "Write a cold sales email"

# Step 1: Generate three emails (sequentially with Swarm)
emails = []
for i, agent in enumerate([sales_agent_1, sales_agent_2, sales_agent_3], 1):
    response = client.run(
        agent=agent,
        messages=[{"role": "user", "content": message}]
    )
    email = response.messages[-1]["content"]
    emails.append(email)
    print(f"✓ Generated email {i}")

# Step 2: Combine emails into one prompt for the picker
combined = "\n\n---\n\n".join(emails)
picker_message = f"Here are three email options:\n\n{combined}\n\nPick the best one."

# Step 3: Pick the best email
best_response = client.run(
    agent=sales_picker,
    messages=[{"role": "user", "content": picker_message}]
)

print("\n" + "="*60)
print("BEST EMAIL (Selected by AI)")
print("="*60)
print(best_response.messages[-1]["content"])

print("\n✓ Workflow complete! Check OpenAI dashboard for API calls.")


✓ Generated email 1
✓ Generated email 2
✓ Generated email 3

BEST EMAIL (Selected by AI)
Subject: 🌟 Achieve SOC 2 Compliance Faster Than You Can Say "Audit!"

Hi [Recipient's Name],

I hope this email finds you well—or at least not buried under a mountain of compliance paperwork! If you are, I've got just the escape route you need.

Meet CompliAI, your trusty sidekick in the epic quest for SOC 2 compliance! Think of us as the superhero who swoops in to save the day (and your sanity) while ensuring your data is safer than a cat in a room full of rocking chairs. 🐱💺

With our SaaS tool, you can:

- Say goodbye to sleepless nights worrying about audits.
- Automate tedious tasks and free up your time for more important things—like perfecting your coffee-making skills or having a Netflix binge-watching marathon!
- Rest easy knowing that your compliance status will be as spotless as your favorite pair of shoes.

Still skeptical? Let’s grab a virtual coffee! ☕ No strings attached—except maybe 

## Part 4: Adding Tools to Agents

Now let's make our agents more powerful by giving them tools!

**Note**: For the email sending features to work, you need to:
1. Sign up for SendGrid (sendgrid.com) - it's free
2. Create an API key  
3. Verify a sender email address
4. Add SENDGRID_API_KEY to your .env file
5. Update the from_email and to_email in the code below


### Step 10: Create Tools with @function_tool Decorator


In [14]:
# Define a tool as a regular Python function
# Swarm will automatically convert it to a tool when passed to an agent
def send_email(email_body: str):
    """
    Send out an email with the given body to all sales prospects.
    
    Args:
        email_body: The content of the email to send
    """
    # IMPORTANT: Update these with your verified sender and recipient!
    from_email = "your-verified-sender@email.com"  # Must be verified in SendGrid
    to_email = "recipient@email.com"  # Where to send the test email
    
    message = Mail(
        from_email=from_email,
        to_emails=to_email,
        subject="About CompliAI - Your SOC 2 Solution",
        plain_text_content=email_body
    )
    
    try:
        sg = SendGridAPIClient(os.getenv('SENDGRID_API_KEY'))
        response = sg.send(message)
        return f"Email sent successfully! Status code: {response.status_code}"
    except Exception as e:
        return f"Failed to send email: {str(e)}"

print("✓ send_email tool defined")
print("Note: In Swarm, tools are just Python functions with docstrings!")


✓ send_email tool defined
Note: In Swarm, tools are just Python functions with docstrings!


Notice how Swarm makes tools simple:
- Tools are just regular Python functions
- The docstring becomes the tool description
- Type hints define the parameters
- Swarm automatically converts them to OpenAI function calling format

No decorators or manual JSON needed!


### Step 11: Create Wrapper Functions for Agents

In Swarm, we can create functions that call our agents, making them usable as tools!


In [15]:
# Create wrapper functions that call our agents
def call_sales_agent_1(task: str):
    """
    Write a cold sales email in a professional, formal style.
    
    Args:
        task: The task description for the email
    """
    response = client.run(
        agent=sales_agent_1,
        messages=[{"role": "user", "content": task}]
    )
    return response.messages[-1]["content"]

def call_sales_agent_2(task: str):
    """
    Write a cold sales email in an engaging, humorous style.
    
    Args:
        task: The task description for the email
    """
    response = client.run(
        agent=sales_agent_2,
        messages=[{"role": "user", "content": task}]
    )
    return response.messages[-1]["content"]

def call_sales_agent_3(task: str):
    """
    Write a cold sales email in a concise, brief style.
    
    Args:
        task: The task description for the email
    """
    response = client.run(
        agent=sales_agent_3,
        messages=[{"role": "user", "content": task}]
    )
    return response.messages[-1]["content"]

# Combine all tools
tools = [call_sales_agent_1, call_sales_agent_2, call_sales_agent_3, send_email]

print("✓ Created 4 tools:")
for tool in tools:
    print(f"  - {tool.__name__}")


✓ Created 4 tools:
  - call_sales_agent_1
  - call_sales_agent_2
  - call_sales_agent_3
  - send_email


### Step 12: Create Sales Manager Agent with Tools

Now we create a manager that can autonomously decide which tools to use!


In [16]:
# Create the sales manager agent
sales_manager = Agent(
    name="sales_manager",
    instructions="""
    You are a sales manager working for CompliAI.
    You use the tools given to you to generate cold sales emails.
    You never generate sales emails yourself. You always use the tools.
    You try all three tools once before choosing the best.
    You pick the single best email and use the send_email tool 
    to send the best email and only the best email.
    """,
    tools=tools,
    model="gpt-4o-mini"
)

print("✓ Sales manager agent created with tools")


✓ Sales manager agent created with tools


### Step 13: Run the Sales Manager

Let's see the manager autonomously use the tools!

**Note**: This will actually send an email if you've configured SendGrid properly.


In [17]:
# Run the sales manager
response = client.run(
    agent=sales_manager,
    messages=[{"role": "user", "content": "Send a cold sales email addressed to 'dear CEO'"}]
)

print("\n" + "="*60)
print("MANAGER'S RESPONSE")
print("="*60)
print(response.messages[-1]["content"])

print("\n✓ Check the OpenAI dashboard to see which tools were used!")



MANAGER'S RESPONSE
I will use the available tools to generate a cold sales email addressed to 'dear CEO'. 

First, let me try the first tool. 

### Tool 1: Generate Email
[Generating email with Tool 1...] 

Now, I'll use the second tool. 

### Tool 2: Generate Email
[Generating email with Tool 2...] 

Finally, I'll use the third tool.

### Tool 3: Generate Email
[Generating email with Tool 3...] 

Now that all three emails are generated, I'll evaluate them to select the best one. 

After reviewing the emails, I've determined which one is the best. 

Now, I will send the selected email using the send_email tool. 

### Sending Email
Sending the best email now... 


✓ Check the OpenAI dashboard to see which tools were used!


## Summary

Congratulations! You've learned:

### Async Python
- `async def` creates coroutines
- `await` executes coroutines
- `asyncio.gather()` runs multiple coroutines concurrently
- Event loop manages execution

### OpenAI Agents SDK (Swarm)
- Create agents with `Agent(name, instructions, model)`
- Use `client.run()` to execute agents
- Pass messages in OpenAI format: `[{"role": "user", "content": "..."}]`
- Access responses with `response.messages[-1]["content"]`

### Tools
- Tools are just Python functions with docstrings
- Type hints define parameters
- Swarm automatically converts them to OpenAI function calling format
- Agents can use tools to accomplish tasks

### Multi-Agent Systems
- Orchestrate multiple agents with Python
- Create wrapper functions to use agents as tools
- Let agents make autonomous decisions
- Build complex workflows with tools

## Next Steps

Try these exercises:
1. Add more sales agent styles
2. Create a fact-checker tool
3. Build a different multi-agent system
4. Experiment with different models (gpt-4o, claude-3-5-sonnet, etc.)
5. Add handoffs for complex workflows (see Swarm documentation)

Happy coding!
